# Ocular dominance - group analysis (v11)

**Simplified version** (decided with the supervisor): one weight, one OD measure, medians everywhere, all the possible splits of the runs.

**v11 = v9 + individual subjects on every bar plot.** Figures 1 and 2 now show one dot per subject on the bars (as Figure 3), with a reference line at H0 (50 % / 0); the y axis of Figure 2 covers every subject. Nothing else changes (same data, same statistics, same caches).

**v9 = v8 + local cache.** The vertex data of each subject (betas of both eyes, pRF parameters and fit p-value of each eye, ROI masks; vertices of the 8 ROIs only) are read once from the data disks and saved in `Desktop/fig_od/od_vertex_cache/`. The next runs read this local cache. A cache file is rebuilt automatically when one of its source files (OD image, pRF derivatives or stats) is newer than the cache. The filters (`select`) are applied after loading, so a change of thresholds in `prf-analysis.yml` does not require rebuilding the cache. Delete the folder to force a full reload.

| | v8 | v7 |
|---|---|---|
| Vertex weight (everywhere) | mean pRF R² of the two eyes | same |
| Vertex selection | prf_analysis filters on both eyes, no filter on β | same |
| Figure 2 | **weighted median of the vertex OD index** `(β_FE − β_AE) / (β_FE + β_AE)` (the value shown on the flatmaps) | ratio of the ROI mean betas |
| Reproducibility | **median over the 3 possible splits** of the runs | split bars / rings + wedges only |
| Negative control | crossed labels, split bars / rings + wedges (the only split where it is valid) | same |

**Story**

1. **How much cortex prefers each eye?** Weighted % of vertices preferring FE-LE, i.e. with a positive OD index (Figure 1).
2. **How strong is the dominance?** Weighted median of the OD index of the vertices of the ROI (Figure 2).
3. **Is it reproducible, and does it come from the eye?** OD maps of two independent halves of the runs, median over the 3 splits (Figure 3A-B); crossed-label negative control (Figure 3C).
4. Supplementary: subject-level replication, biases, between-group comparison, diagnostics.

**Vertex OD index.** `(β_FE − β_AE) / (β_FE + β_AE)` is only meaningful when both betas are positive: with a denominator close to 0 it explodes, and with one negative beta its sign can be wrong (β_FE = 0.2, β_AE = −0.5 gives −2.3 although the fellow eye responds more). On the 4 runs, fewer than 0.5 % of the vertices have a β ≤ 0 (reported below), so the vertex OD index is computed where both betas are > 0 and the excluded vertices are negligible. Figure 1 keeps every vertex (the vote β_FE > β_AE is valid for any sign; for the vertices with both betas > 0 it is exactly "OD index > 0").

**Maps of the halves (Figure 3).** With 2 runs per half, the betas are noisier and 10-25 % of the vertices of V1-V2 have a β ≤ 0 in at least one half: the vertex OD index is unstable there. The maps of the halves therefore use the same numerator with the denominator of the ROI: `(β_FE − β_AE) / (β̄_FE + β̄_AE)` (β̄ = weighted ROI mean in that half), defined for every vertex.

**Why the mean pRF R².** The weight comes from the pRF fits, an earlier and independent step: it does not depend on the betas used to measure the dominance. It is the usual weight of the pRF analyses.

**Conventions:** FE-LE = fellow eye (amblyopes) / left eye (controls); AE-RE = amblyopic eye / right eye. Bars = median across subjects, error bars = 25th-75th percentiles. Stars = test within each group (amblyopes one-sided in the direction of the hypothesis, controls two-sided unless stated), FDR over the 8 ROIs: \* p < 0.05, \*\* < 0.01, \*\*\* < 0.001. Non-parametric tests (Wilcoxon) because of small groups (12, 8), bounded measures and a few atypical subjects. Nothing is written in the data folders; figures go to `Desktop/fig_od` (suffix `_v11`).

In [6]:
# Stop warnings
import warnings
warnings.filterwarnings("ignore")

# General imports
import os
import sys
import numpy as np
import pandas as pd
from scipy import stats, integrate
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Personal imports
sys.path.append("{}/../../analysis_code/utils".format(os.getcwd()))
from settings_utils import load_settings
from pycortex_utils import set_pycortex_config_file, get_rois, load_surface_pycortex
from plot_utils import plotly_template

In [7]:
# Directories
main_dir = '/Users/uriel/disks/meso_shared'
project_dir = 'amblyo7T_prf'
pp_dir = '{}/{}/derivatives/pp_data'.format(main_dir, project_dir)
fig_dir = '/Users/uriel/Desktop/fig_od'
cache_dir = '/Users/uriel/Desktop/fig_od/od_glm_common_cache'       # per-run GLM sums (od_glm_common_analysis.ipynb)
os.makedirs(fig_dir, exist_ok=True)

# Settings
base_dir = os.path.abspath(os.path.join(os.getcwd(), "../../"))
settings = load_settings([os.path.join(base_dir, project_dir, f) for f in
                          ["settings.yml", "prf-analysis.yml", "figure-settings.yml"]])
analysis_info = settings[0]
patients = analysis_info['group-patient']
controls = analysis_info['group-control']
subjects = patients + controls                 # analysed subjects = the two groups (not analysis_info['subjects'])
maps_names_css = analysis_info['maps_names_css']
maps_names_css_stats = analysis_info['maps_names_css_stats']
maps_names_css_od = analysis_info['maps_names_css_od']

# Analysis
task = 'BarsBarsRingsWedges'
file_spec = '{}/{}/fsnative/prfmodel/prf_derivatives/{}_task-{}{}_hemi-{}_fmriprep_dct_psc_concat_prf-css_{}.func.gii'
rois_to_keep = ['V1', 'V2', 'V3', 'V4', 'V3A', 'V3B', 'LO1', 'LO2']
eyes = {'L': 'LeftEye', 'R': 'RightEye'}

# prf_analysis filters (applied to both eyes)
ecc_th = analysis_info['ecc_th']
size_th = analysis_info['size_th']
n_th = analysis_info['n_th']
rsqr_th = analysis_info['rsqr_th']
amp_th = analysis_info['prf_amp_th']
amp_th = amp_th['psc'] if isinstance(amp_th, dict) else amp_th
stats_th = analysis_info['stats_th']
p_name = 'corr_pvalue_5pt' if stats_th == 0.05 else 'corr_pvalue_1pt'

# Amblyopic eye of each patient
participants = pd.read_csv('{}/{}/participants.tsv'.format(main_dir, project_dir), sep='\t')
ambly_eye = {s: str(e).split(' ')[0] for s, e in zip(participants.participant_id, participants.amblyopic_eye)
             if str(e).split(' ')[0] in ('L', 'R')}

# Eye scanned first (read only: sourcedata/participant-mapping.md + participant_mapping.tsv)
src_dir = '{}/{}/sourcedata'.format(main_dir, project_dir)
mapping = pd.read_csv('{}/participant_mapping.tsv'.format(src_dir), sep='\t', dtype=str).dropna(subset=['initials'])
initials_to_sub = {i.strip(): b.strip() for i, b in zip(mapping.initials, mapping.bids_name)}
first_eye = {}
with open('{}/participant-mapping.md'.format(src_dir)) as f:
    for line in f:
        if 'eye first' in line:
            initials, txt = line.split(',', 1)
            first_eye[initials_to_sub[initials.strip()]] = 'L' if 'LEFT' in txt.upper() else 'R'

# Pycortex database (ROIs)
cortex_dir = '{}/{}/derivatives/pp_data/cortex'.format(main_dir, project_dir)
set_pycortex_config_file(cortex_dir)

In [3]:
def bh_fdr(p):
    """Benjamini-Hochberg FDR correction."""
    p = np.asarray(p, float)
    o = np.argsort(p)
    q = p[o] * len(p) / (np.arange(len(p)) + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[o] = np.minimum(q, 1)
    return out


def wcorr(x, y, w):
    """Weighted rank correlation: Pearson correlation of the ranks of x and y, each vertex weighted by w."""
    x, y, w = stats.rankdata(x), stats.rankdata(y), np.asarray(w, float)
    if np.sum(w) == 0:
        return np.nan
    mx, my = np.sum(w * x) / np.sum(w), np.sum(w * y) / np.sum(w)
    return np.sum(w * (x - mx) * (y - my)) / np.sqrt(np.sum(w * (x - mx) ** 2) * np.sum(w * (y - my) ** 2))


def stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''


def bf10_ttest(x, h0=0, r=np.sqrt(2) / 2):
    """JZS Bayes factor BF10 of a one-sample t-test (Rouder et al. 2009), Cauchy prior of scale r on the effect size."""
    x = np.asarray(x, float)
    x = x[np.isfinite(x)] - h0
    n, v = x.size, x.size - 1
    t = x.mean() / (x.std(ddof=1) / np.sqrt(n))
    null = (1 + t ** 2 / v) ** (-(v + 1) / 2)
    alt = integrate.quad(lambda g: (1 + n * g * r ** 2) ** -0.5
                         * (1 + t ** 2 / ((1 + n * g * r ** 2) * v)) ** (-(v + 1) / 2)
                         * (2 * np.pi) ** -0.5 * g ** -1.5 * np.exp(-1 / (2 * g)), 0, np.inf)[0]
    return alt / null


def within_tests(df, value, h0, alt_patient='greater', alt_control='two-sided'):
    """Per ROI: Wilcoxon of each group vs h0 (FDR over the ROIs) + Bayes factor BF01 of the controls."""
    rows = []
    for roi in rois_to_keep:
        d = df[df.roi == roi]
        vp, vc = d[d.group == 'patient'][value].dropna(), d[d.group == 'control'][value].dropna()
        rows.append({'roi': roi, 'median_patients': vp.median(), 'median_controls': vc.median(),
                     'p_patients': stats.wilcoxon(vp - h0, alternative=alt_patient).pvalue,
                     'p_controls': stats.wilcoxon(vc - h0, alternative=alt_control).pvalue,
                     'BF01_controls': 1 / bf10_ttest(vc, h0)})
    s = pd.DataFrame(rows).set_index('roi')
    s['p_patients_fdr'] = bh_fdr(s.p_patients)
    s['p_controls_fdr'] = bh_fdr(s.p_controls)
    return s


# Figure settings (as the other pRF figures)
template_specs = dict(axes_color="rgba(0, 0, 0, 1)", axes_width=2, axes_font_size=15,
                      bg_col="rgba(255, 255, 255, 1)", font='Arial', title_font_size=15, rois_plot_width=1.5)
fig_template = plotly_template(template_specs)
roi_colors_rgb = analysis_info['roi_colors']
fig_margin = analysis_info['rois_fig_margin']
bar_width = analysis_info['rois_bar_width']
rois_plot_height = analysis_info['rois_plot_height']
rois_hor_spacing = 50
ctrl_color = 'rgb(150,150,150)'
groups = ['patient', 'control']
group_labels = {'patient': 'Amblyopes', 'control': 'Controls'}
cols = len(rois_to_keep)
fig_height = rois_plot_height * 2 + fig_margin[1] + fig_margin[3]
fig_width = bar_width * 4 * cols + fig_margin[0] + fig_margin[2] + (rois_hor_spacing * (cols - 1))
hor_spacing = rois_hor_spacing / (fig_width - fig_margin[0] - fig_margin[2])


def bar_figure(df, values, s, y_title, y_ticks, tick_fmt, fn_out, legends=None, patterns=None, ref=None):
    """Median and 25-75 percentiles per group and ROI + one dot per subject (as Figure 3);
    stars = within-group test of s (FDR); ref = reference line (H0)."""
    patterns = patterns or [''] * len(values)
    rng_dots = np.random.default_rng(0)                                   # same jitter at each run
    w = 0.7 / len(values)
    offs = (np.arange(len(values)) - (len(values) - 1) / 2) * w
    y_step = (y_ticks[-1] - y_ticks[0]) * 0.05
    fig = make_subplots(rows=1, cols=cols, subplot_titles=rois_to_keep, horizontal_spacing=hor_spacing)
    for col_idx, roi in enumerate(rois_to_keep, start=1):
        d = df[df.roi == roi]
        xref, yref = 'x{}'.format('' if col_idx == 1 else col_idx), 'y{}'.format('' if col_idx == 1 else col_idx)
        if ref is not None:
            fig.add_shape(type='line', x0=0, x1=1, y0=ref, y1=ref, xref='{} domain'.format(xref), yref=yref,
                          line=dict(color='grey', width=1))
        for g_i, group in enumerate(groups):
            color = roi_colors_rgb[roi] if group == 'patient' else ctrl_color
            top = 0
            for value, pattern, off in zip(values, patterns, offs):
                v = d[d.group == group][value].dropna()
                med, p25, p75 = v.median(), np.percentile(v, 25), np.percentile(v, 75)
                fig.add_trace(go.Bar(x=[g_i + off], y=[med], width=w * 0.9, showlegend=False,
                                     marker=dict(color=color, pattern=dict(shape=pattern, fgcolor='rgba(0,0,0,1)',
                                                                           bgcolor='rgba(0,0,0,0)', size=8)),
                                     error_y=dict(type='data', array=[p75 - med], arrayminus=[med - p25], visible=True,
                                                  thickness=2, width=0, color='rgba(0,0,0,1)')),
                              row=1, col=col_idx)
                fig.add_trace(go.Scatter(x=g_i + off + rng_dots.uniform(-w * 0.2, w * 0.2, len(v)), y=v, mode='markers',
                                         showlegend=False, marker=dict(size=5, color='rgba(0,0,0,0.6)')),
                              row=1, col=col_idx)
                top = max(top, v.max())
            p = s.loc[roi, 'p_{}_fdr'.format('patients' if group == 'patient' else 'controls')]
            fig.add_annotation(x=g_i, y=min(top + y_step / 2, y_ticks[-1] + y_step), xref=xref, yref=yref, text=stars(p),
                               showarrow=False, font=dict(size=16, color='black'), yanchor='bottom')
        fig.add_shape(type='line', x0=0, x1=0, y0=y_ticks[0], y1=y_ticks[-1], xref='{} domain'.format(xref), yref=yref,
                      line=dict(color=template_specs['axes_color'], width=template_specs['axes_width']))
    if legends:
        for legend, pattern in zip(legends, patterns):
            fig.add_trace(go.Bar(x=[None], y=[None], name=legend,
                                 marker=dict(color='rgba(128,128,128,1)',
                                             pattern=dict(shape=pattern, fgcolor='rgba(0,0,0,1)',
                                                          bgcolor='rgba(0,0,0,0.3)', size=8)),
                                 showlegend=True), row=1, col=1)
    fig.update_xaxes(showline=True, ticklen=0, linecolor='rgba(255,255,255,0)', tickangle=0, range=[-0.6, 1.6],
                     tickvals=[0, 1], ticktext=[group_labels[g] for g in groups])
    fig.update_yaxes(showline=False, range=[y_ticks[0], y_ticks[-1] + 3 * y_step], tickvals=y_ticks,
                     ticktext=[tick_fmt.format(t) for t in y_ticks], showticklabels=True)
    fig.update_yaxes(title_text=y_title, row=1, col=1)
    fig.update_layout(height=fig_height, width=fig_width, template=fig_template,
                      legend=dict(orientation='h', font_family=template_specs['font'],
                                  font_size=template_specs['axes_font_size'], x=1, y=1.15, xanchor='right',
                                  yanchor='bottom', bgcolor='rgba(0,0,0,0)', traceorder='normal', itemwidth=30),
                      margin_l=fig_margin[0], margin_t=fig_margin[1] + 40, margin_r=fig_margin[2], margin_b=fig_margin[3])
    fig.show()
    fig.write_image('{}/{}.pdf'.format(fig_dir, fn_out))


def nice_ticks(values, step):
    lo = min(0, np.floor(np.nanmin(values) / step) * step)
    hi = max(step, np.ceil(np.nanmax(values) / step) * step)
    return np.round(np.arange(lo, hi + step / 2, step), 3)


def wpearson(x, y, w):
    """Weighted Pearson correlation (= correlation of a weighted linear regression)."""
    x, y, w = np.asarray(x, float), np.asarray(y, float), np.asarray(w, float)
    if np.sum(w) == 0 or len(x) < 3:
        return np.nan
    mx, my = np.sum(w * x) / np.sum(w), np.sum(w * y) / np.sum(w)
    return np.sum(w * (x - mx) * (y - my)) / np.sqrt(np.sum(w * (x - mx) ** 2) * np.sum(w * (y - my) ** 2))


def deming(x, y, w=None):
    """Weighted Deming (orthogonal) regression, as in make_corr_tsv.py: returns slope, intercept.
    Symmetric in x and y (both halves are measured with noise), weights normalised to sum 1."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    w = np.ones_like(x) if w is None else np.asarray(w, float)
    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(w)
    x, y, w = x[ok], y[ok], w[ok]
    if len(x) < 3 or np.sum(w) == 0:
        return np.nan, np.nan
    w = w / np.sum(w)
    mx, my = np.sum(w * x), np.sum(w * y)
    sxx, syy, sxy = np.sum(w * (x - mx) ** 2), np.sum(w * (y - my) ** 2), np.sum(w * (x - mx) * (y - my))
    if sxy == 0:
        return np.nan, np.nan
    slope = (syy - sxx + np.sqrt((syy - sxx) ** 2 + 4 * sxy ** 2)) / (2 * sxy)
    return slope, my - slope * mx


def wmedian(x, w):
    """Weighted median: first value where the cumulative weight reaches half of the total."""
    x, w = np.asarray(x, float), np.asarray(w, float)
    ok = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if ok.sum() == 0:
        return np.nan
    o = np.argsort(x[ok])
    cw = np.cumsum(w[ok][o])
    return x[ok][o][np.searchsorted(cw, 0.5 * cw[-1])]


def od_index(b_fe, b_ae):
    """OD index (FE - AE) / (FE + AE); defined for positive betas only."""
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where((b_fe > 0) & (b_ae > 0), (b_fe - b_ae) / (b_fe + b_ae), np.nan)

## Load the data: measures per subject and ROI

Per subject and ROI (vertices kept: prf_analysis filters on both eyes, all betas kept; weight w = mean pRF R² of both eyes):

- `pref_FE-LE` / `pref_AE-RE`: weighted % of vertices preferring each eye, `100 · Σ w·[β_FE > β_AE] / Σ w` (Figure 1);
- `od_index`: **weighted median of the vertex OD index** `(β_FE − β_AE) / (β_FE + β_AE)` over the vertices of the ROI with both betas > 0 (Figure 2): the typical value of the flatmap in this ROI;
- % of the vertices (and of the weight) with β ≤ 0 for the FE-LE eye only, the AE-RE eye only, or both: these vertices are kept in Figure 1 and excluded from the vertex OD index (Figure 2);
- `pref_fe_sel` / `pref_ae_sel`, `rsq_fe` / `rsq_ae`: vertex-selection checks (supplementary).

In [4]:
# ---------- local cache of the vertex data (vertices of the 8 ROIs only)
vcache_dir = '{}/od_vertex_cache'.format(fig_dir)
os.makedirs(vcache_dir, exist_ok=True)
deriv_keys = ['prf_rsq', 'prf_ecc', 'prf_size', 'prf_n', 'amplitude', 'prf_x', 'prf_y']


def subject_files(subject):
    fns = [file_spec.format(pp_dir, subject, subject, task, '', hemi, 'od') for hemi in ['L', 'R']]
    for eye in eyes.values():
        for kind in ['deriv', 'stats']:
            fns += [file_spec.format(pp_dir, subject, subject, task, eye, hemi, kind) for hemi in ['L', 'R']]
    return fns


def load_subject(subject):
    """Vertex data of one subject: betas, pRF parameters and fit p-value of each eye, ROI masks (full-length arrays).
    Read from the local cache when it is newer than all the source files, otherwise from the data disks (then cached)."""
    fns = subject_files(subject)
    if not all(os.path.isfile(fn) for fn in fns):
        return None
    cache_fn = '{}/{}_od-vertex-cache.npz'.format(vcache_dir, subject)
    src_mtime = max(os.path.getmtime(fn) for fn in fns)
    if not (os.path.isfile(cache_fn) and float(np.load(cache_fn)['src_mtime']) >= src_mtime):
        od_mat = load_surface_pycortex(L_fn=fns[0], R_fn=fns[1])['data_concat']
        rois = get_rois(subject=subject, surf_format='fsnative', rois_type='rois-mmp', mask=True, rois=None, hemis=None)
        idx = np.where(np.any([rois[roi] for roi in rois_to_keep], axis=0))[0]
        save = {'n_vertices': od_mat.shape[1], 'idx': idx, 'src_mtime': src_mtime,
                'beta_L': od_mat[maps_names_css_od.index('od_beta_le')][idx].astype(np.float32),
                'beta_R': od_mat[maps_names_css_od.index('od_beta_re')][idx].astype(np.float32)}
        for roi in rois_to_keep:
            save['roi_{}'.format(roi)] = rois[roi][idx]
        for e, eye in eyes.items():
            deriv = load_surface_pycortex(L_fn=file_spec.format(pp_dir, subject, subject, task, eye, 'L', 'deriv'),
                                          R_fn=file_spec.format(pp_dir, subject, subject, task, eye, 'R', 'deriv'))['data_concat']
            st = load_surface_pycortex(L_fn=file_spec.format(pp_dir, subject, subject, task, eye, 'L', 'stats'),
                                       R_fn=file_spec.format(pp_dir, subject, subject, task, eye, 'R', 'stats'))['data_concat']
            for key in deriv_keys:
                save['{}_{}'.format(e, key)] = deriv[maps_names_css.index(key)][idx].astype(np.float32)
            save['{}_p'.format(e)] = st[maps_names_css_stats.index(p_name)][idx].astype(np.float32)
        np.savez_compressed(cache_fn, **save)
        print('{}: read from the data disks, cached'.format(subject), end=' | ')
    c = np.load(cache_fn)
    n, idx = int(c['n_vertices']), c['idx']

    def full(x, fill=np.nan):
        out = np.full(n, fill, dtype=float if fill is not False else bool)
        out[idx] = x
        return out
    return {'beta': {e: full(c['beta_{}'.format(e)]) for e in eyes},
            'd': {e: {key: full(c['{}_{}'.format(e, key)]) for key in deriv_keys} for e in eyes},
            'p': {e: full(c['{}_p'.format(e)]) for e in eyes},
            'rois': {roi: full(c['roi_{}'.format(roi)], False) for roi in rois_to_keep}}


def select(d, p):
    """prf_analysis filters of one eye + positive pRF amplitude (p = p-value of the pRF fit)."""
    with np.errstate(invalid='ignore'):
        return ((d['prf_ecc'] >= ecc_th[0]) & (d['prf_ecc'] <= ecc_th[1])
                & (d['prf_size'] >= size_th[0]) & (d['prf_size'] <= size_th[1])
                & (d['prf_n'] >= n_th[0]) & (d['prf_n'] <= n_th[1])
                & (d['prf_rsq'] >= rsqr_th)
                & (d['amplitude'] >= amp_th[0]) & (d['amplitude'] <= amp_th[1]) & (d['amplitude'] > 0)
                & (p <= stats_th))


def pref_percent(b_a, b_b, w):
    return 100 * np.sum(w * (b_a > b_b)) / np.sum(w) if np.sum(w) > 0 else np.nan


rows, vertex_data = [], {}
for subject in subjects:
    group = 'patient' if subject in patients else 'control' if subject in controls else None
    sd = load_subject(subject)
    if sd is None:
        print('{}: OD image or pRF files missing, skipped'.format(subject))
        continue
    beta = sd['beta']
    keep_eye = {e: select(sd['d'][e], sd['p'][e]) & np.isfinite(beta[e]) for e in eyes}
    rsq_eye = {e: sd['d'][e]['prf_rsq'] for e in eyes}
    weight = np.nan_to_num(np.clip(np.nanmean([rsq_eye['L'], rsq_eye['R']], axis=0), 0, None))   # mean pRF R² of both eyes

    # FE-LE / AE-RE: amblyopes -> fellow / amblyopic eye, controls -> left / right eye
    fe, ae = ('R', 'L') if ambly_eye.get(subject) == 'L' else ('L', 'R')
    b_fe, b_ae = beta[fe], beta[ae]
    keep = keep_eye['L'] & keep_eye['R']                            # pRF filters on both eyes, all betas kept
    vertex_data[subject] = {'fe': fe, 'keep': keep, 'w': weight}

    rois = sd['rois']
    for roi in rois_to_keep:
        mask = rois[roi] & keep
        w = weight[mask]
        if mask.sum() < 10 or np.sum(w) == 0:
            continue
        mask_fe, mask_ae = rois[roi] & keep_eye[fe], rois[roi] & keep_eye[ae]
        rows.append({'subject': subject, 'group': group, 'roi': roi, 'n_vertices': mask.sum(),
                     'fe_le_first': first_eye[subject] == fe,
                     # vertices with beta <= 0 (kept in Figure 1, excluded from the vertex OD index)
                     'neg_any_pct': 100 * np.mean((b_fe[mask] <= 0) | (b_ae[mask] <= 0)),
                     'neg_any_w_pct': 100 * np.sum(w * ((b_fe[mask] <= 0) | (b_ae[mask] <= 0))) / np.sum(w),
                     'neg_fe_only_pct': 100 * np.mean((b_fe[mask] <= 0) & (b_ae[mask] > 0)),
                     'neg_ae_only_pct': 100 * np.mean((b_ae[mask] <= 0) & (b_fe[mask] > 0)),
                     'neg_both_pct': 100 * np.mean((b_fe[mask] <= 0) & (b_ae[mask] <= 0)),
                     # measures
                     'pref_FE-LE': pref_percent(b_fe[mask], b_ae[mask], w),
                     'pref_AE-RE': pref_percent(b_ae[mask], b_fe[mask], w),
                     'od_index': wmedian(od_index(b_fe[mask], b_ae[mask]), w),     # weighted median of the vertex OD index (both β > 0)
                     # vertex-selection checks
                     'pref_fe_sel': pref_percent(b_fe[mask_fe], b_ae[mask_fe], weight[mask_fe]),
                     'pref_ae_sel': pref_percent(b_fe[mask_ae], b_ae[mask_ae], weight[mask_ae]),
                     'rsq_fe': np.nanmedian(rsq_eye[fe][mask]), 'rsq_ae': np.nanmedian(rsq_eye[ae][mask])})
    print('{}: done'.format(subject))
df_subjects = pd.DataFrame(rows)
df_subjects.groupby(['group', 'roi'])[['n_vertices', 'pref_FE-LE', 'od_index']].median().unstack('group').reindex(rois_to_keep).round(3)

sub-01: read from the data disks, cached | sub-01: done
sub-02: read from the data disks, cached | sub-02: done
sub-03: read from the data disks, cached | sub-03: done
sub-04: read from the data disks, cached | sub-04: done
sub-05: read from the data disks, cached | sub-05: done
sub-06: read from the data disks, cached | sub-06: done
sub-07: read from the data disks, cached | sub-07: done
sub-08: read from the data disks, cached | sub-08: done
sub-09: read from the data disks, cached | sub-09: done
sub-10: read from the data disks, cached | sub-10: done
sub-11: read from the data disks, cached | sub-11: done
sub-12: read from the data disks, cached | sub-12: done
sub-13: read from the data disks, cached | sub-13: done
sub-14: read from the data disks, cached | sub-14: done
sub-16: read from the data disks, cached | sub-16: done
sub-17: read from the data disks, cached | sub-17: done
sub-18: read from the data disks, cached | sub-18: done
sub-19: read from the data disks, cached | sub-1

n_vertices         pref_FE-LE         od_index        
group    control patient    control patient  control patient
roi                                                         
V1        6031.0  5385.0     51.746  57.805    0.012   0.020
V2        6172.0  5520.0     55.484  63.862    0.037   0.084
V3        4159.0  3714.0     59.754  71.518    0.051   0.132
V4        2958.0  2906.0     58.727  70.698    0.041   0.102
V3A       1069.0  1285.0     64.072  70.224    0.027   0.126
V3B        481.0   611.0     40.247  81.659   -0.049   0.120
LO1        527.0   544.0     55.240  76.963    0.017   0.136
LO2        429.0   359.0     67.820  66.974    0.076   0.126

### Vertices with β ≤ 0 (kept in Figure 1, excluded from the vertex OD index)

Median across subjects, % of the selected vertices (and of the weight) with β ≤ 0 for at least one eye, for the FE-LE eye only, the AE-RE eye only, or both. A vertex with β_AE ≤ 0 < β_FE is a vertex where the amblyopic eye does not drive the cortex: it prefers the fellow eye. The last column tests, in the amblyopes, whether β ≤ 0 is more frequent for the AE than for the FE (paired Wilcoxon, two-sided, FDR).

In [ ]:
cols_rm = ['neg_any_pct', 'neg_any_w_pct', 'neg_fe_only_pct', 'neg_ae_only_pct', 'neg_both_pct']
rm = df_subjects.groupby(['group', 'roi'])[cols_rm].median().round(1)
pat = df_subjects[df_subjects.group == 'patient']


def safe_wilcoxon(x, y):
    """Paired Wilcoxon; NaN when all the differences are zero (nothing to test)."""
    d = np.asarray(x) - np.asarray(y)
    if np.all(d == 0):
        return np.nan
    return stats.wilcoxon(x, y).pvalue

p_rm = [safe_wilcoxon(pat[pat.roi == roi].neg_ae_only_pct, pat[pat.roi == roi].neg_fe_only_pct) for roi in rois_to_keep]
ok = ~np.isnan(p_rm)
p_fdr = np.full(len(p_rm), np.nan)
p_fdr[ok] = bh_fdr(np.array(p_rm)[ok])
rm_p = pd.Series(p_fdr, index=rois_to_keep, name='p_fdr AE vs FE (amblyopes)').round(3)
print(rm.unstack('group').reindex(rois_to_keep).to_string())
print()
print(rm_p.to_string())

## Figure 1 — how much cortex prefers each eye

**Measure:** weighted % of vertices preferring the FE-LE eye. 50 % = no preference.

**Amblyopes:** Wilcoxon vs 50 %, one-sided (FE > AE), FDR over the 8 ROIs. **Controls:** the hypothesis is the absence of dominance; a non-significant test does not show it, so we add the **Bayes factor BF01** (JZS t-test, Cauchy prior r = 0.707): BF01 > 3 = moderate evidence for no dominance, 1-3 = anecdotal, < 1 = favours a dominance. Two-sided Wilcoxon vs 50 %.

Each vertex votes for one eye, whatever the size of its preference: this says how much cortex prefers each eye, not how strongly (Figure 2).

Bars = median across subjects, error bars = 25th-75th percentiles, dots = individual subjects; grey line = H0.

In [ ]:
s_pref = within_tests(df_subjects, 'pref_FE-LE', 50)
print('% of vertices preferring FE-LE (H0 = 50 %): amblyopes one-sided, controls two-sided')
s_pref[['median_patients', 'p_patients_fdr', 'median_controls', 'p_controls_fdr', 'BF01_controls']].round(3)

In [ ]:
bar_figure(df_subjects, ['pref_FE-LE', 'pref_AE-RE'], s_pref, 'Preferring vertices (%)', np.linspace(0, 100, 5), '{:.0f}',
           'od_v11_fig1_pref_percent', legends=['FE / LE', 'AE / RE'], patterns=['', '/'], ref=50)

## Figure 2 — how strong is the dominance

**Measure:** for each vertex with both betas > 0, OD index `(β_FE − β_AE) / (β_FE + β_AE)` (the value shown on the flatmaps); per subject and ROI, **weighted median** over the vertices (weight = mean pRF R²). 0 = both eyes respond equally; +0.1 = the fellow eye responds 1.22 times more than the amblyopic eye ((1 + OD) / (1 − OD)); bounded between −1 and 1 and symmetric for the two eyes. The median is robust to the vertices with a very weak response, whose OD index is close to ±1.

**Test:** Wilcoxon vs 0 (amblyopes one-sided, controls two-sided + BF01), FDR over the ROIs.

Bars = median across subjects, error bars = 25th-75th percentiles, dots = individual subjects; grey line = H0.

In [ ]:
s_od = within_tests(df_subjects, 'od_index', 0)
print('Weighted median of the vertex OD index (H0 = 0): amblyopes one-sided, controls two-sided')
s_od[['median_patients', 'p_patients_fdr', 'median_controls', 'p_controls_fdr', 'BF01_controls']].round(3)

In [ ]:
bar_figure(df_subjects, ['od_index'], s_od, 'OD index, median (FE-LE − AE-RE)',
           nice_ticks(df_subjects.od_index, 0.1), '{:.1f}',
           'od_v11_fig2_od_index', ref=0)

## Figure 3 — reproducibility of the OD maps and negative control

**Idea (as Nasr et al. 2025 and de Hollander et al. 2021):** split the runs into two independent halves, compute the OD map in each half, and check that the second half finds the map of the first one. If the betas were random, the two maps would not be related.

**All the splits.** Each half must contain both eyes; with 4 runs per eye (Bars01, Bars02, Rings, Wedges) there are exactly **3** ways to make two pairs of runs:

| split | half A | half B |
|---|---|---|
| 1 | Bars01 + Bars02 | Rings + Wedges |
| 2 | Bars01 + Rings | Bars02 + Wedges |
| 3 | Bars01 + Wedges | Bars02 + Rings |

Every measure is computed for each split, then the **median over the 3 splits** is taken per subject and ROI. Splits 2 and 3 have the same stimulus composition in both halves (closer to two identical sessions, as Nasr et al.); split 1 compares different stimuli (more demanding).

**OD map of a half:** for each vertex, `(β_FE − β_AE) / (β̄_FE + β̄_AE)`, with the betas of the 2 runs of the half (per-run sums of `od_glm_common_analysis.ipynb`): same numerator as the vertex OD index, denominator of the ROI in that half (β̄ = weighted mean β of the ROI), so that it is defined for every vertex despite the noisier betas of the halves. Same vertices as above.

**Reproducibility:** for each subject, ROI and split, **weighted linear regression** of the map of half B on the map of half A, weighted by the mean pRF R² (as `make_corr_tsv.py`): weighted Pearson r (the statistic tested) and weighted **Deming** slope (orthogonal regression, symmetric because both halves contain noise). r > 0 = a vertex that prefers the fellow eye in one half also prefers it in the other; slope ≈ 1 = same amplitude in both halves. Test: median r over the 3 splits > 0 (Wilcoxon, one-sided, FDR over the ROIs, per group).

**Negative control: crossed labels (split 1 only).** In each half, pseudo-eye 1 = first run of the FE eye + second run of the AE eye, pseudo-eye 2 = the reverse: stimuli, regressor, vertices and amount of data are identical, but the real eye difference cancels. If the reproducible maps come from the eye, the crossed r is ~0. This control is only valid in **split 1**: with crossed labels, the map of a half is "dominance of its first run − dominance of its second run"; in half A of split 1 both runs are bars, so this map contains only run-to-run noise. In splits 2 and 3, both halves would compare "bars − other stimulus" and share the part of the dominance that depends on the stimulus, so the crossed r would not be 0 for a reason unrelated to the eye. The control therefore compares true and crossed labels **within split 1** (Figure 3C).

**Limit:** both halves contain runs of each eye scanned in the same order: a systematic order effect would also replicate (tested in S2). **Not done:** blind-spot control (stimulus ±10°, blind spot ~15°).

In [ ]:
splits = {1: ([0, 1], [2, 3]), 2: ([0, 2], [1, 3]), 3: ([0, 3], [1, 2])}     # run order in the cache: Bars01, Bars02, Rings, Wedges


def od_map(b_fe, b_ae, m, w):
    """OD map of a ROI: (β_FE − β_AE) of each vertex / (β̄_FE + β̄_AE) of the ROI (weighted means)."""
    den = np.sum(w * b_fe[m]) / np.sum(w) + np.sum(w * b_ae[m]) / np.sum(w)
    return (b_fe[m] - b_ae[m]) / den


split_rows, sh_vertex = [], {}
for subject in df_subjects.subject.unique():
    cache_fn = '{}/{}_od-glm-common_runs.npz'.format(cache_dir, subject)
    if not os.path.isfile(cache_fn):
        print('{}: no per-run cache, skipped'.format(subject))
        continue
    cache = np.load(cache_fn)
    vd = vertex_data[subject]
    vert = cache['vertex']
    keep, w_all, roi_names = vd['keep'][vert], vd['w'][vert], cache['roi']
    fe_i, ae_i = (1, 0) if vd['fe'] == 'R' else (0, 1)            # cache eye order: 0 = left, 1 = right
    sxy, sxx = cache['sxy_common'], cache['sxx_common']
    group = 'patient' if subject in patients else 'control' if subject in controls else None

    def beta(items):
        """β of a group of (eye, run) items (same regressor for both eyes in a run)."""
        with np.errstate(divide='ignore', invalid='ignore'):
            return sum(sxy[e, r] for e, r in items) / sum(sxx[r] for _, r in items)

    sh_vertex[subject] = {'keep': keep, 'roi': roi_names, 'w': w_all, 'b': {}}
    for split, (runs_A, runs_B) in splits.items():
        b, b_cross = {}, {}
        for half, (r1, r2) in zip(['A', 'B'], [runs_A, runs_B]):
            b[half] = (beta([(fe_i, r1), (fe_i, r2)]), beta([(ae_i, r1), (ae_i, r2)]))
            b_cross[half] = (beta([(fe_i, r1), (ae_i, r2)]), beta([(ae_i, r1), (fe_i, r2)]))
        sh_vertex[subject]['b'][split] = b
        finite = np.all([np.isfinite(x) for h in 'AB' for x in b[h] + b_cross[h]], axis=0)
        for roi in rois_to_keep:
            m = keep & (roi_names == roi) & finite
            w = w_all[m]
            if m.sum() < 10 or np.sum(w) == 0:
                continue
            od = {h: od_map(*b[h], m, w) for h in 'AB'}
            slope, _ = deming(od['A'], od['B'], w)
            row = {'subject': subject, 'group': group, 'roi': roi, 'split': split, 'n_vertices': m.sum(),
                   'r_true': wpearson(od['A'], od['B'], w), 'deming_slope': slope}
            # ROI OD index of each half (same measure as Figure 2: weighted median of the vertex OD index, both β > 0 in that half)
            for h in 'AB':
                pos = (b[h][0][m] > 0) & (b[h][1][m] > 0)
                row['od_{}'.format(h)] = wmedian(od_index(b[h][0][m], b[h][1][m]), w)
                row['excluded_{}_pct'.format(h)] = 100 * (1 - pos.mean())
            if split == 1:
                od_c = {h: od_map(*b_cross[h], m, w) for h in 'AB'}
                row['r_crossed'] = wpearson(od_c['A'], od_c['B'], w)
            split_rows.append(row)
df_split = pd.DataFrame(split_rows)

# Median over the 3 splits per subject and ROI (r_crossed: split 1 only)
df_sh = df_split.groupby(['subject', 'group', 'roi'])[['r_true', 'deming_slope']].median().reset_index()
df_sh = df_sh.merge(df_split[df_split.split == 1][['subject', 'roi', 'r_true', 'r_crossed']]
                    .rename(columns={'r_true': 'r_true_split1'}), on=['subject', 'roi'], how='left')

print('Median r per split (medians across subjects):')
print(df_split.groupby(['group', 'roi', 'split']).r_true.median().unstack(['group', 'split']).reindex(rois_to_keep).round(2).to_string())

# Tests: median-of-3 r > 0 (one-sided); split 1: true r > crossed r (one-sided), crossed r vs 0 (two-sided); FDR over the ROIs
rows = []
for roi in rois_to_keep:
    for group in groups:
        dg = df_sh[(df_sh.roi == roi) & (df_sh.group == group)]
        dc = dg.dropna(subset=['r_true_split1', 'r_crossed'])
        rows.append({'roi': roi, 'group': group, 'n': len(dg), 'r_median3': dg.r_true.median(), 'slope_median3': dg.deming_slope.median(),
                     'r_split1': dc.r_true_split1.median(), 'r_crossed_split1': dc.r_crossed.median(),
                     'p_r_vs_0': stats.wilcoxon(dg.r_true.dropna(), alternative='greater').pvalue,
                     'p_true_vs_crossed': stats.wilcoxon(dc.r_true_split1, dc.r_crossed, alternative='greater').pvalue,
                     'p_crossed_vs_0': stats.wilcoxon(dc.r_crossed).pvalue})
sh_stats = pd.DataFrame(rows)
for col in ['p_r_vs_0', 'p_true_vs_crossed', 'p_crossed_vs_0']:
    sh_stats[col + '_fdr'] = sh_stats.groupby('group')[col].transform(bh_fdr)
sh_stats = sh_stats.set_index(['group', 'roi'])
print('\nMap reproducibility (medians across subjects)')
print(sh_stats[['n', 'r_median3', 'slope_median3', 'p_r_vs_0_fdr', 'r_split1', 'r_crossed_split1',
                'p_true_vs_crossed_fdr', 'p_crossed_vs_0_fdr']].round(3).to_string())

### Figure 3A — example subjects

OD map value of each vertex in half A (x) and half B (y), V1-V3, for each of the 3 splits pooled in one panel (colours = splits; r and Deming slope = median over the 3 splits), for a representative amblyope and a representative control (the subject of each group whose median-of-3 r averaged over V1-V3 is the closest to the group median: not the best subjects). Dot size ∝ weight; dashed = identity.

In [ ]:
early = ['V1', 'V2', 'V3']
mean_r = df_sh[df_sh.roi.isin(early)].groupby(['group', 'subject']).r_true.mean()
examples = {g: (mean_r[g] - mean_r[g].median()).abs().idxmin() for g in groups}
print('Representative subjects:', examples)
split_colors = {1: 'rgba(31,119,180,0.35)', 2: 'rgba(255,127,14,0.35)', 3: 'rgba(44,160,44,0.35)'}

fig = make_subplots(rows=2, cols=3, horizontal_spacing=0.08, vertical_spacing=0.2,
                    subplot_titles=['{} {} - {}'.format(group_labels[g][:-1], examples[g], roi) for g in groups for roi in early])
for r_i, group in enumerate(groups, start=1):
    sv = sh_vertex[examples[group]]
    for c_i, roi in enumerate(early, start=1):
        n = (r_i - 1) * 3 + c_i
        xref, yref = 'x{}'.format('' if n == 1 else n), 'y{}'.format('' if n == 1 else n)
        allv = []
        for split in splits:
            b = sv['b'][split]
            m = sv['keep'] & (sv['roi'] == roi) & np.isfinite(b['A'][0]) & np.isfinite(b['B'][0])
            w_v = sv['w'][m]
            x_v, y_v = od_map(*b['A'], m, w_v), od_map(*b['B'], m, w_v)
            allv.append(np.concatenate([x_v, y_v]))
            fig.add_trace(go.Scattergl(x=x_v, y=y_v, mode='markers', name='split {}'.format(split),
                                       showlegend=(n == 1), legendgroup=str(split),
                                       marker=dict(size=2 + 6 * w_v / np.max(w_v), color=split_colors[split])), row=r_i, col=c_i)
        lim = np.nanpercentile(np.abs(np.concatenate(allv)), 99)
        fig.add_shape(type='line', x0=-lim, x1=lim, y0=-lim, y1=lim, xref=xref, yref=yref, line=dict(color='grey', dash='dash'))
        d = df_sh[(df_sh.subject == examples[group]) & (df_sh.roi == roi)].iloc[0]
        fig.add_annotation(x=0.04, y=0.97, xref='{} domain'.format(xref), yref='{} domain'.format(yref), xanchor='left',
                           yanchor='top', showarrow=False, font=dict(size=14),
                           text='r = {:.2f}<br>slope = {:.2f}'.format(d.r_true, d.deming_slope))
        fig.update_xaxes(range=[-lim, lim], row=r_i, col=c_i)
        fig.update_yaxes(range=[-lim, lim], row=r_i, col=c_i)
fig.update_xaxes(title_text='OD map, half A', showline=True, zeroline=True, zerolinecolor='lightgrey')
fig.update_yaxes(title_text='OD map, half B', showline=True, zeroline=True, zerolinecolor='lightgrey')
fig.update_layout(template=fig_template, height=2 * rois_plot_height * 1.4 + fig_margin[1] + fig_margin[3],
                  width=3 * rois_plot_height * 1.4 + fig_margin[0] + fig_margin[2],
                  legend=dict(orientation='h', x=1, y=1.12, xanchor='right', yanchor='bottom', bgcolor='rgba(0,0,0,0)'),
                  margin_l=fig_margin[0], margin_t=fig_margin[1] + 40, margin_r=fig_margin[2], margin_b=fig_margin[3])
fig.show()
fig.write_image('{}/od_v11_fig3A_split_half_maps.pdf'.format(fig_dir))

### Figure 3B — map reproducibility, all subjects

For each subject and ROI, median over the 3 splits of the weighted r between the OD maps of the two halves. Median and 25-75 %, one dot per subject. Stars: r > 0 (Wilcoxon, one-sided, FDR over the 8 ROIs, per group).

In [ ]:
rng = np.random.default_rng(0)


def dot_bar_figure(conds, star_col, y_title, fn_out):
    """One bar per condition and group (median, 25-75 %) + one dot per subject; stars from sh_stats[star_col]."""
    w = 0.7 / len(conds)
    offs = (np.arange(len(conds)) - (len(conds) - 1) / 2) * w
    r_ticks = nice_ticks(df_sh[[c[0] for c in conds]].values.ravel(), 0.2)
    y_step = (r_ticks[-1] - r_ticks[0]) * 0.05
    fig = make_subplots(rows=1, cols=cols, subplot_titles=rois_to_keep, horizontal_spacing=hor_spacing)
    for col_idx, roi in enumerate(rois_to_keep, start=1):
        for g_i, group in enumerate(groups):
            dg = df_sh[(df_sh.roi == roi) & (df_sh.group == group)]
            color = roi_colors_rgb[roi] if group == 'patient' else ctrl_color
            for (value, pattern, _), off in zip(conds, offs):
                v = dg[value].dropna()
                med, p25, p75 = v.median(), np.percentile(v, 25), np.percentile(v, 75)
                fig.add_trace(go.Bar(x=[g_i + off], y=[med], width=w * 0.9, showlegend=False,
                                     marker=dict(color=color, pattern=dict(shape=pattern, fgcolor='rgba(0,0,0,1)', bgcolor='rgba(0,0,0,0)', size=8)),
                                     error_y=dict(type='data', array=[p75 - med], arrayminus=[med - p25], thickness=2, width=0, color='black')),
                              row=1, col=col_idx)
                fig.add_trace(go.Scatter(x=g_i + off + rng.uniform(-w * 0.2, w * 0.2, len(v)), y=v, mode='markers', showlegend=False,
                                         marker=dict(size=5, color='rgba(0,0,0,0.6)')), row=1, col=col_idx)
            top = max(np.nanmax(dg[[c[0] for c in conds]].values), 0)
            fig.add_annotation(x=g_i, y=min(top + y_step / 2, r_ticks[-1] + y_step), yanchor='bottom', showarrow=False, font=dict(size=16),
                               text=stars(sh_stats.loc[(group, roi), star_col]), row=1, col=col_idx)
        xref, yref = 'x{}'.format('' if col_idx == 1 else col_idx), 'y{}'.format('' if col_idx == 1 else col_idx)
        fig.add_shape(type='line', x0=0, x1=1, y0=0, y1=0, xref='{} domain'.format(xref), yref=yref, line=dict(color='grey', width=1))
        fig.add_shape(type='line', x0=0, x1=0, y0=r_ticks[0], y1=r_ticks[-1], xref='{} domain'.format(xref), yref=yref,
                      line=dict(color=template_specs['axes_color'], width=template_specs['axes_width']))
    if len(conds) > 1:
        for _, pattern, label in conds:
            fig.add_trace(go.Bar(x=[None], y=[None], name=label, marker=dict(color='rgba(128,128,128,1)',
                                 pattern=dict(shape=pattern, fgcolor='rgba(0,0,0,1)', bgcolor='rgba(0,0,0,0.3)', size=8))), row=1, col=1)
    fig.update_xaxes(showline=True, ticklen=0, linecolor='rgba(255,255,255,0)', tickangle=0, range=[-0.6, 1.6],
                     tickvals=[0, 1], ticktext=[group_labels[g] for g in groups])
    fig.update_yaxes(showline=False, range=[r_ticks[0], r_ticks[-1] + 3 * y_step], tickvals=r_ticks,
                     ticktext=['{:.1f}'.format(t) for t in r_ticks])
    fig.update_yaxes(title_text=y_title, row=1, col=1)
    fig.update_layout(template=fig_template, height=fig_height, width=fig_width,
                      legend=dict(orientation='h', font_family=template_specs['font'], font_size=template_specs['axes_font_size'],
                                  x=1, y=1.15, xanchor='right', yanchor='bottom', bgcolor='rgba(0,0,0,0)', itemwidth=30),
                      margin_l=fig_margin[0], margin_t=fig_margin[1] + 40, margin_r=fig_margin[2], margin_b=fig_margin[3])
    fig.show()
    fig.write_image('{}/{}.pdf'.format(fig_dir, fn_out))


dot_bar_figure([('r_true', '', 'True eyes')], 'p_r_vs_0_fdr', 'OD map correlation between halves (r, median of 3 splits)',
               'od_v11_fig3B_map_correlation')

### Figure 3C — negative control (split 1: bars vs rings + wedges)

Same r, in split 1 only, for the true eyes (solid) and the crossed labels (hatched). Stars: true r > crossed r (paired Wilcoxon, one-sided, FDR over the 8 ROIs, per group); the test crossed r vs 0 is in the table above.

In [ ]:
dot_bar_figure([('r_true_split1', '', 'True eyes'), ('r_crossed', '/', 'Crossed labels (control)')], 'p_true_vs_crossed_fdr',
               'OD map correlation, split 1 (r)', 'od_v11_fig3C_negative_control')

# Supplementary

## S1 — subject-level replication

OD index of each subject (same measure as Figure 2: weighted median of the vertex OD index, vertices with both β > 0 **in that half**) in half A vs half B. Per split: number of amblyopes with OD > 0 in both halves and Pearson r across the 20 subjects (each subject with the same weight); then median over the 3 splits. The figure shows the 3 splits (symbols) with the Deming line of all points. The % of vertices excluded by β ≤ 0 in the halves is printed (it is larger than on the 4 runs: noisier betas).

In [ ]:
print('Vertices with β ≤ 0 in a half (excluded from the vertex OD index of S1), median %:')
print(df_split.assign(excl=df_split[['excluded_A_pct', 'excluded_B_pct']].mean(axis=1))
      .groupby(['group', 'roi']).excl.median().unstack('group').reindex(rois_to_keep).round(1).to_string())

rows = []
for roi in rois_to_keep:
    per_split = []
    for split in splits:
        d = df_split[(df_split.roi == roi) & (df_split.split == split)].dropna(subset=['od_A', 'od_B'])
        dp = d[d.group == 'patient']
        per_split.append({'both_pos': np.sum((dp.od_A > 0) & (dp.od_B > 0)), 'n': len(dp),
                          'r': stats.pearsonr(d.od_A, d.od_B)[0], 'med_A': dp.od_A.median(), 'med_B': dp.od_B.median()})
    ps = pd.DataFrame(per_split)
    rows.append({'roi': roi, 'pat_both_halves_pos (median of 3)': '{:.0f}/{}'.format(ps.both_pos.median(), int(ps.n.max())),
                 'pat_median_A': ps.med_A.median(), 'pat_median_B': ps.med_B.median(),
                 'r_subjects (median of 3)': ps.r.median(), 'r_subjects per split': ' / '.join('{:.2f}'.format(x) for x in ps.r)})
print(pd.DataFrame(rows).set_index('roi').round(3).to_string())

symbols = {1: 'circle', 2: 'square', 3: 'diamond'}
vals = df_split[['od_A', 'od_B']].values.ravel()
lo, hi = min(-0.1, np.floor(np.nanpercentile(vals, 1) * 10) / 10), np.ceil(np.nanpercentile(vals, 99) * 10) / 10
fig = make_subplots(rows=1, cols=cols, subplot_titles=rois_to_keep, horizontal_spacing=hor_spacing)
for col_idx, roi in enumerate(rois_to_keep, start=1):
    dr = df_split[df_split.roi == roi].dropna(subset=['od_A', 'od_B'])
    xref, yref = 'x{}'.format('' if col_idx == 1 else col_idx), 'y{}'.format('' if col_idx == 1 else col_idx)
    fig.add_shape(type='line', x0=lo, x1=hi, y0=lo, y1=hi, xref=xref, yref=yref, line=dict(color='grey', width=1, dash='dash'))
    fig.add_shape(type='line', x0=lo, x1=hi, y0=0, y1=0, xref=xref, yref=yref, line=dict(color='lightgrey', width=1))
    fig.add_shape(type='line', x0=0, x1=0, y0=lo, y1=hi, xref=xref, yref=yref, line=dict(color='lightgrey', width=1))
    for group in groups:
        for split in splits:
            dg = dr[(dr.group == group) & (dr.split == split)]
            fig.add_trace(go.Scatter(x=dg.od_A, y=dg.od_B, mode='markers', name='{} - split {}'.format(group_labels[group], split),
                                     showlegend=col_idx == 1,
                                     marker=dict(size=8, symbol=symbols[split], color=roi_colors_rgb[roi] if group == 'patient' else ctrl_color,
                                                 line=dict(width=1, color='black'))), row=1, col=col_idx)
    slope, intercept = deming(dr.od_A, dr.od_B)
    fig.add_trace(go.Scatter(x=[lo, hi], y=[intercept + slope * lo, intercept + slope * hi], mode='lines',
                             line=dict(color='black', width=1.5), showlegend=False), row=1, col=col_idx)
    r_med = np.median([stats.pearsonr(dr[dr.split == sp].od_A, dr[dr.split == sp].od_B)[0] for sp in splits])
    fig.add_annotation(x=0.05, y=0.97, xref='{} domain'.format(xref), yref='{} domain'.format(yref), xanchor='left', yanchor='top',
                       text='r = {:.2f}'.format(r_med), showarrow=False, font=dict(size=14))
fig.update_xaxes(range=[lo, hi], showline=True, tickangle=0)
fig.update_yaxes(range=[lo, hi], showline=True)
fig.update_xaxes(title_text='Half A', row=1, col=1)
fig.update_yaxes(title_text='Half B: OD index', row=1, col=1)
fig.update_layout(template=fig_template, height=fig_height, width=fig_width,
                  legend=dict(orientation='h', font_family=template_specs['font'], font_size=12,
                              x=1, y=1.15, xanchor='right', yanchor='bottom', bgcolor='rgba(0,0,0,0)'),
                  margin_l=fig_margin[0], margin_t=fig_margin[1] + 40, margin_r=fig_margin[2], margin_b=fig_margin[3])
fig.show()
fig.write_image('{}/od_v11_S1_subject_replication.pdf'.format(fig_dir))

## S2 — biases

**Eye order.** Each eye was scanned in separate runs: a global change between the two parts of the session could look like a dominance. Amblyopes: % FE and OD index of the subjects with FE scanned first (n = 4) vs second (n = 8) (Mann-Whitney, two-sided). Controls: % of vertices preferring the eye scanned **first** vs 50 % (Wilcoxon, two-sided).

**Vertex selection.** Amblyopes: median pRF R² of FE vs AE (paired Wilcoxon); % FE recomputed with the vertices selected on the pRF filters of the fellow eye only and of the amblyopic eye only (Wilcoxon vs 50 %, one-sided).

**Circularity** was tested in `od_glm_common_analysis.ipynb` (section 11): regressing each eye on the **other** eye's prediction gives the same results.

In [ ]:
rows = []
for roi in rois_to_keep:
    d = df_subjects[df_subjects.roi == roi]
    pat, ctrl = d[d.group == 'patient'], d[d.group == 'control']
    first_pref = np.where(ctrl.fe_le_first, ctrl['pref_FE-LE'], ctrl['pref_AE-RE'])
    rows.append({'roi': roi,
                 # Eye order
                 'pat_pref_FE_first': pat[pat.fe_le_first]['pref_FE-LE'].median(),
                 'pat_pref_FE_second': pat[~pat.fe_le_first]['pref_FE-LE'].median(),
                 'p_order_pref': stats.mannwhitneyu(pat[pat.fe_le_first]['pref_FE-LE'], pat[~pat.fe_le_first]['pref_FE-LE']).pvalue,
                 'p_order_od': stats.mannwhitneyu(pat[pat.fe_le_first].od_index.dropna(), pat[~pat.fe_le_first].od_index.dropna()).pvalue,
                 'ctrl_pref_first_eye': np.median(first_pref),
                 'p_ctrl_first_eye': stats.wilcoxon(first_pref - 50).pvalue,
                 # Vertex selection
                 'pat_rsq_fe': pat.rsq_fe.median(), 'pat_rsq_ae': pat.rsq_ae.median(),
                 'p_rsq_fe_vs_ae': stats.wilcoxon(pat.rsq_fe, pat.rsq_ae).pvalue,
                 'pat_pref_sel_fe': pat.pref_fe_sel.median(),
                 'p_sel_fe': stats.wilcoxon(pat.pref_fe_sel - 50, alternative='greater').pvalue,
                 'pat_pref_sel_ae': pat.pref_ae_sel.median(),
                 'p_sel_ae': stats.wilcoxon(pat.pref_ae_sel - 50, alternative='greater').pvalue})
bias = pd.DataFrame(rows).set_index('roi')
for col in ['p_order_pref', 'p_order_od', 'p_ctrl_first_eye', 'p_rsq_fe_vs_ae', 'p_sel_fe', 'p_sel_ae']:
    bias[col + '_fdr'] = bh_fdr(bias[col])
print('Eye order')
print(bias[['pat_pref_FE_first', 'pat_pref_FE_second', 'p_order_pref_fdr', 'p_order_od_fdr',
            'ctrl_pref_first_eye', 'p_ctrl_first_eye_fdr']].round(3))
print('\nVertex selection (amblyopes)')
print(bias[['pat_rsq_fe', 'pat_rsq_ae', 'p_rsq_fe_vs_ae_fdr', 'pat_pref_sel_fe', 'p_sel_fe_fdr',
            'pat_pref_sel_ae', 'p_sel_ae_fdr']].round(3))

## S3 — amblyopes vs controls (exploratory)

Direct comparison (Mann-Whitney, two-sided, FDR). With 12 vs 8 subjects it has little power.

In [ ]:
rows = []
for roi in rois_to_keep:
    d = df_subjects[df_subjects.roi == roi]
    row = {'roi': roi}
    for value in ['pref_FE-LE', 'od_index']:
        vp, vc = d[d.group == 'patient'][value].dropna(), d[d.group == 'control'][value].dropna()
        row['diff_{}'.format(value)] = vp.median() - vc.median()
        row['p_{}'.format(value)] = stats.mannwhitneyu(vp, vc).pvalue
    rows.append(row)
between = pd.DataFrame(rows).set_index('roi')
for value in ['pref_FE-LE', 'od_index']:
    between['p_{}_fdr'.format(value)] = bh_fdr(between['p_{}'.format(value)])
between.round(3)

## S4 — diagnostics: which subjects have non-reproducible maps?

Map correlation (true eyes, median of the 3 splits, Figure 3B) per subject and ROI, and its relation with descriptors of the data: number of vertices, pRF R² (minimum of both eyes), shift of the pRF position between the eyes (in degrees and relative to the pRF size), eccentricity and size differences. Empirical noise spread = SD of the crossed-label r of each ROI (split 1); the median of 3 splits is less noisy than one split, so the criterion is conservative.

In [ ]:
import matplotlib.pyplot as plt

df_b = df_sh.copy()
subj_order = [s for s in patients + controls if s in df_b.subject.unique()]

# Empirical noise spread per ROI: crossed-label correlations (no eye effect)
null_sd = df_b.groupby('roi').r_crossed.std().reindex(rois_to_keep)
df_b['below_noise'] = df_b.r_true < -2 * df_b.roi.map(null_sd)
print('SD of the crossed-label r per ROI (noise spread):')
print(null_sd.round(3).to_string())
print('\nSubject x ROI with r_true < -2 SD of the noise spread:')
print(df_b[df_b.below_noise][['subject', 'group', 'roi', 'r_true']].round(3).to_string(index=False))

# Heatmap subjects x ROIs
piv = df_b.pivot(index='subject', columns='roi', values='r_true').reindex(index=subj_order, columns=rois_to_keep)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(piv.values, cmap='RdBu_r', vmin=-0.7, vmax=0.7, aspect='auto')
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        v = piv.values[i, j]
        if np.isfinite(v):
            ax.text(j, i, '{:.2f}'.format(v), ha='center', va='center', fontsize=7)
ax.set_xticks(range(len(rois_to_keep)))
ax.set_xticklabels(rois_to_keep)
ax.set_yticks(range(len(subj_order)))
ax.set_yticklabels(['{} ({})'.format(s, 'A' if s in patients else 'C') for s in subj_order])
ax.axhline(sum(s in patients for s in subj_order) - 0.5, color='k', lw=1.5)
plt.colorbar(im, ax=ax, label='map correlation, bars vs rings + wedges (r)')
ax.set_title('Map correlation per subject and ROI (A = amblyope, C = control)')
plt.tight_layout()
fig.savefig('{}/od_v11_S4_r_heatmap.pdf'.format(fig_dir), bbox_inches='tight')
plt.show()

In [ ]:
# Descriptors per subject and ROI: number of vertices, fit quality, difference between the pRFs of the two eyes
rows = []
for subject in subj_order:
    vd = vertex_data[subject]
    fe = vd['fe']
    ae = 'L' if fe == 'R' else 'R'
    sd = load_subject(subject)                                      # local cache
    d, rois = sd['d'], sd['rois']
    dist = np.hypot(d[fe]['prf_x'] - d[ae]['prf_x'], d[fe]['prf_y'] - d[ae]['prf_y'])
    for roi in rois_to_keep:
        m = rois[roi] & vd['keep']
        rows.append({'subject': subject, 'roi': roi, 'n_vertices': m.sum(),
                     'rsq_min': np.nanmedian(np.minimum(d[fe]['prf_rsq'][m], d[ae]['prf_rsq'][m])),
                     'pos_shift_deg': np.nanmedian(dist[m]),
                     'pos_shift_rel': np.nanmedian(dist[m] / ((d[fe]['prf_size'][m] + d[ae]['prf_size'][m]) / 2)),
                     'ecc_diff_deg': np.nanmedian(d[ae]['prf_ecc'][m] - d[fe]['prf_ecc'][m]),
                     'size_ratio_ae_fe': np.nanmedian(d[ae]['prf_size'][m] / d[fe]['prf_size'][m]),
                     'ecc': np.nanmedian((d[fe]['prf_ecc'][m] + d[ae]['prf_ecc'][m]) / 2)})
    print('{}: done'.format(subject))
diag = pd.DataFrame(rows).merge(df_b[['subject', 'roi', 'group', 'r_true', 'r_crossed', 'below_noise']], on=['subject', 'roi'])

# Which descriptor goes with low / negative r? Spearman across subject x ROI (all, amblyopes, controls)
descs = ['n_vertices', 'rsq_min', 'pos_shift_deg', 'pos_shift_rel', 'ecc_diff_deg', 'size_ratio_ae_fe', 'ecc']
res = []
for desc in descs:
    row = {'descriptor': desc}
    for name, dd in [('all', diag), ('amblyopes', diag[diag.group == 'patient']), ('controls', diag[diag.group == 'control'])]:
        dd = dd.dropna(subset=[desc, 'r_true'])
        rho, p = stats.spearmanr(dd[desc], dd.r_true)
        row['rho_{}'.format(name)], row['p_{}'.format(name)] = rho, p
    res.append(row)
print(pd.DataFrame(res).set_index('descriptor').round(3).to_string())

# Same, removing the ROI effect (each descriptor and r centred per ROI): does it hold within ROIs?
dc = diag.copy()
for col in descs + ['r_true']:
    dc[col] = dc[col] - dc.groupby('roi')[col].transform('median')
print('\nWithin ROIs (values centred per ROI):')
print(pd.DataFrame([{'descriptor': desc, 'rho': stats.spearmanr(dc[desc], dc.r_true, nan_policy='omit')[0],
                     'p': stats.spearmanr(dc[desc], dc.r_true, nan_policy='omit')[1]} for desc in descs]).set_index('descriptor').round(3).to_string())

In [ ]:
# Scatter plots: r vs each descriptor (one dot per subject and ROI; amblyopes in colour, controls in grey)
fig, axes = plt.subplots(2, 4, figsize=(16, 7.5))
for ax, desc in zip(axes.ravel(), descs):
    for group, color in [('patient', '#d95f02'), ('control', '0.55')]:
        dd = diag[diag.group == group]
        ax.scatter(dd[desc], dd.r_true, s=18, color=color, alpha=0.8, label=group_labels[group])
        neg = dd[dd.below_noise]
        ax.scatter(neg[desc], neg.r_true, s=40, facecolors='none', edgecolors='k')
    ax.axhline(0, color='grey', lw=0.8)
    ax.set_xlabel(desc)
    ax.set_ylabel('map correlation (r)')
    if desc in ['n_vertices']:
        ax.set_xscale('log')
axes.ravel()[-1].axis('off')
axes.ravel()[0].legend(frameon=False, fontsize=8)
plt.suptitle('Map correlation vs descriptors (circled: r below −2 SD of the crossed-label noise)')
plt.tight_layout()
fig.savefig('{}/od_v11_S4_r_vs_descriptors.pdf'.format(fig_dir), bbox_inches='tight')
plt.show()

# Per subject: mean r over the ROIs vs mean pRF shift between the eyes
per_subj = diag.groupby(['subject', 'group'])[['r_true', 'pos_shift_rel', 'rsq_min']].mean().reset_index()
print(per_subj.sort_values('r_true').round(3).to_string(index=False))
print('Amblyopes, subject mean r vs pRF shift: rho = {:.2f}, p = {:.3f}'.format(
    *stats.spearmanr(per_subj[per_subj.group == 'patient'].pos_shift_rel, per_subj[per_subj.group == 'patient'].r_true)))